Load the MNIST dataset

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import datetime

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Normalize pixel values to the range [0, 1].

In [2]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

Flatten the 28 x 28 images into 1-D vectors of length 784

In [3]:
X_train = X_train.reshape(-1, 28*28)
X_test = X_test.reshape(-1, 28*28)

Build a baseline ANN model using Dense layers.

ReLU (Hidden Layers): It prevents the vanishing gradient problem. This ensures the gradients do not shrink to zero during backpropagation, allowing the optimizer to effectively update the weights.

Softmax (Output Layer): It converts the raw network outputs into a strict probability distribution. It ensures the predictions across the 10 distinct digit classes are between 0 and 1, and sum to exactly 1.

In [4]:
def build_model(lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

Compile the model with an optimizer, loss function, and metric.

In [5]:
def build_model(lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

Train the model while manually logging hyperparameters and results.

In [6]:
experiment_log = []

def run_experiment(exp_id, lr, epochs, batch_size):
    model = build_model(lr)

    start = datetime.datetime.now()

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    duration = (datetime.datetime.now() - start).total_seconds()

    experiment_log.append({
        'exp_id': exp_id,
        'learning_rate': lr,
        'epochs': epochs,
        'batch_size': batch_size,
        'val_accuracy': round(max(history.history['val_accuracy']), 4),
        'test_accuracy': round(test_acc, 4),
        'training_time_sec': round(duration, 2)
    })

    return model

In [7]:
run_experiment('EXP-01', lr=0.001, epochs=10, batch_size=32)
run_experiment('EXP-02', lr=0.01, epochs=10, batch_size=32)
run_experiment('EXP-03', lr=0.0001, epochs=10, batch_size=64)

log_df = pd.DataFrame(experiment_log)
log_df.to_csv('experiment_log.csv', index=False)
print(log_df)

   exp_id  learning_rate  epochs  batch_size  val_accuracy  test_accuracy  \
0  EXP-01         0.0010      10          32        0.9818         0.9792   
1  EXP-02         0.0100      10          32        0.9702         0.9651   
2  EXP-03         0.0001      10          64        0.9690         0.9628   

   training_time_sec  
0              54.04  
1              52.01  
2              31.00  
